# Inspect the Blackboard

This example notebook demonstrates the following:

- Inspect and modify runtime values on the current blackboard
- Modify loop counters via the blackboard
- Create and load blackboard snapshots

<div class="alert alert-info">

**Important**

This notebook requires a running Flowstate solution to connect to. To start a solution:

1. Navigate to [flowstate.intrinsic.ai](https://flowstate.intrinsic.ai/) and sign in
   using your registered Flowstate account.

1. Do **one** of the following:
    - Create a new solution:
        1. Click "Create new solution" and choose "From an example".
        1. Select `pick_and_place:pick_and_place_module2`
        1. Click "Create".
    - Or open an existing solution that was created from the `pick_and_place:pick_and_place_module2` example:
        1. Hover over the solution in the list.
        1. Click "Open solution" or "Start solution".

1. Recommended: Keep the browser tab with the Flowstate solution editor open to watch the effect of notebook actions such as running a skill. You can simultaneously interact with the solution through the web UI and the notebook.

</div>

First, connect to your solution and define convenience shortcuts:

In [ ]:
from intrinsic.solutions import behavior_tree as bt
from intrinsic.solutions import deployments

solution = deployments.connect_to_selected_solution()

executive = solution.executive
skills = solution.skills
world = solution.world

move_robot = skills.ai.intrinsic.move_robot
estimate_pose = skills.ai.intrinsic.estimate_pose

## Work with blackboard values

### A simple process that writes results
Here we are interested in the result of the `estimate_pose` skill that detects a block

In [ ]:
move_left = bt.Task(name="Move Left", action=move_robot(
    motion_segments=[
        move_robot.intrinsic_proto.skills.MotionSegment(
            joint_position=world.robot.joint_configurations.view_pose_left,
            motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
        )
    ],
    arm_part=world.robot,
))
estimate_block = bt.Task(name="Estimate Block", action=estimate_pose(
    camera=solution.resources.wrist_camera,
    pose_estimator=solution.pose_estimators.building_block_ml_estimator,
    return_value_key="estimate_block_result"
))
executive.run(bt.BehaviorTree(root=bt.Sequence(children=[move_left, estimate_block])))

Access to the current operation's blackboard is available via `executive.operation.blackboard`.

We can list current values on the blackboard.
Note that this also shows an entry from move_robot. In this example we care about the return value from estimate_pose.

In [ ]:
op = executive.operation

op.blackboard.list_keys()

We can view the contents of a specific entry by passing in the skill call's `return_value_key` to `get_value`.

In [ ]:
op.blackboard.get_value('estimate_block_result')

If we have a reference to the skill, we can also just refer to its result

In [ ]:
op.blackboard.get_value(estimate_block.result)

Finally we can delete a key

In [ ]:
op.blackboard.delete_value(estimate_block.result)
op.blackboard.list_keys()

<div class="alert alert-info">

**Warning**

Any modification of values on the blackboard can lead to unexpected results during execution

## Blackboard values as counters

As a simple example process consider this loop node with a maximum count of `5` and a simple breakpoint node.

In [ ]:
loop = bt.Loop(max_times=5, loop_counter_key="my_loop_counter",
                do_child=bt.Sequence([bt.Debug(name="Break")]))
executive.run(bt.BehaviorTree(root=loop))

We observe that the 'loop_counter' is found on the blackboard

In [ ]:
op = executive.operation
op.blackboard.list_keys()

It's current value is shown as a number. Here 0 as we are in the first iteration.

In [ ]:
op.blackboard.get_value(loop.loop_counter)

Resuming moves the loop into the second iteration, which has the index 1.

The counter can also be accessed by its name directly

In [ ]:
executive.resume()
executive.block_until_completed()

op.blackboard.get_value("my_loop_counter")

It is also possible to update values on the blackboard.

If the value is a loop or retry node's counter this will also affect the node's counter during execution.

In [ ]:
op.blackboard.update_value(loop.loop_counter, 4)

Resuming now will forward the loop to its final iteration

In [ ]:
executive.resume()
executive.block_until_completed()
print(op.done)

<div class="alert alert-info">

**Warning**

Modifying values and especially counters should only be done for debugging and development as this can lead to unexpected results during execution

## Blackboard snapshots
Blackboard snapshots record the current state of the operation's blackboard.

The values from the blackboard are extracted and stored independent of the operation and thus will persist deleting the current operation or starting a new one.

All values are stored in memory on the executive. As such it is important to delete snapshots if they are not required any more to keep the memory footprint within limits.

#### Example
We will use a simple loop with a breakpoint node and a pose estimation.

Note that the breakpoint is entered at the start of a loop iteration.

In [ ]:
loop = bt.Loop(max_times=5, loop_counter_key="my_loop_counter",
                do_child=bt.Sequence([bt.Debug(name="Break"), move_left, estimate_block]))
executive.run(bt.BehaviorTree(root=loop))
op = executive.operation

We performing two more iterations.

This leads to the loop being at the start of the third iteration with index 2.

In [ ]:
executive.resume()
executive.block_until_completed()
executive.resume()
executive.block_until_completed()
op.blackboard.get_value(loop.loop_counter)

We now take a snapshot of the current values on the blackboard

In [ ]:
snapshot = op.blackboard.create_snapshot(display_name="My snapshot")

Now we load a completely new operation

In [ ]:
executive.run(bt.BehaviorTree(root=loop))
op = executive.operation

It's count starts at 0

In [ ]:
op.blackboard.get_value(loop.loop_counter)

Its blackboard also only has the counter value as the breakpoint is at the start of the iteration

In [ ]:
op.blackboard.list_keys()

We can now load our stored snapshot

In [ ]:
load_diagnostics = op.blackboard.load_snapshot(snapshot)

<div class="alert alert-info">

**Important**

A snapshot is restored by merging the stored values into the current operation's blackboard.
For these to affect the current operation it must also refer to the same values in its process. Otherwise these just sit there.

This is usually the case when loading the same process that the snapshot was taken with, although this is not a requirement.

All keys from before are there without any skills having been executed

In [ ]:
op.blackboard.list_keys()

Also the loop counter has been forwarded to the iteration when the snapshot was taken

In [ ]:
op.blackboard.get_value(loop.loop_counter)

We can show the currently stored snapshots showing our snapshot with the name `My snapshot`

In [ ]:
executive.blackboard_snapshots.list()

When it is not required any more we can delete this snapshot

In [ ]:
executive.blackboard_snapshots.delete(snapshot)
executive.blackboard_snapshots.list()